<div dir="rtl" align="right">

# التصنيفُ بِالانحدارِ اللوجستيِّ

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُدرّبُ مُصنّفَ انحدارٍ لوجستيٍّ على سماتِ قُوّةِ النطاقِ ونُقيّمُهُ بِمصفوفةِ الالتباسِ وتحليلِ المُعاملاتِ.

## ماذا يَعمَلُ هذا الدفترُ؟

يَحسبُ سماتِ قُوّةِ النطاقِ، ويُقسّمُ إلى تدريب/اختبار، ويُقيّسُ السماتِ، ويُدرّبُ الانحدارَ اللوجستيَّ، ويَطبعُ الدقّةَ.

## المُخرجاتُ المُتوقّعةُ

- مصفوفةُ الالتباسِ تُظهرُ الوسومَ الحقيقيّةَ مقابلَ المُتوقّعةَ
- مخططٌ شريطيٌّ لِمُعاملاتِ النموذجِ مُرتّبةٍ حسبَ القيمةِ المُطلقةِ
- أهمُّ 10 مُعاملاتٍ مُميّزةٌ بِالأخضرِ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| max_iter | 1000 |
| test_size | 0.2 |

</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تدريبُ الانحدارِ اللوجستيِّ

</div>


In [ ]:
from scipy.signal import welch

FS = 250
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

print(f'Feature matrix shape: {features.shape}')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=['left_hand', 'right_hand'])

print(f'Accuracy: {accuracy:.4f}')
print(f'Confusion matrix:\n{cm}')

coefficients = clf.coef_[0]
classes = ['left_hand', 'right_hand']


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- مصفوفةُ الالتباسِ تُظهرُ كمْ تجربةً مُصنّفةٌ بشكلٍ صحيحٍ مقابلَ غيرِ صحيحٍ
- المُعاملاتُ الكبيرةُ المُطلقةُ تُشيرُ إلى سماتٍ مُهمّةٍ
- أهمُّ 10 سماتٍ تُساهمُ أكثرَ في حدِّ القرارِ

</div>


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sorted_idx = np.argsort(np.abs(coefficients))[::-1]
top_10 = set(sorted_idx[:10])
colors = ['green' if i in top_10 else 'steelblue' for i in range(len(coefficients))]

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Confusion Matrix',
    'Model Coefficients (top 10 in green)'))

fig.add_trace(go.Heatmap(z=cm, x=classes, y=classes, colorscale='Blues',
    text=cm, texttemplate='%{text}', textfont={'size': 16}, name='CM', showscale=True), row=1, col=1)
fig.add_trace(go.Bar(x=[str(i) for i in sorted_idx], y=coefficients[sorted_idx],
    marker_color=colors, name='Coefficient'), row=2, col=1)

fig.update_xaxes(title_text='Predicted', row=1, col=1)
fig.update_yaxes(title_text='True', row=1, col=1)
fig.update_xaxes(title_text='Feature (sorted by |coefficient|)', row=2, col=1)
fig.update_yaxes(title_text='Coefficient', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='Logistic Regression Classification')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- الانحدارُ اللوجستيُّ يُوفّرُ مُعاملاتٍ قابلةً لِلتفسيرِ لِكلِّ سمةٍ
- التقييسُ ضروريٌّ قبلَ التدريبِ لِضمانِ مُقارنةٍ عادلةٍ لِلمُعاملاتِ
- مصفوفةُ الالتباسِ تَكشفُ أيُّ الفئاتِ تَلتبسُ معَ بعضِها

</div>
